# Project 4: Scraping Central Bank Communications
## A Structured Dataset of Central-Bank Communications

**Tool:** Python (requests, BeautifulSoup, Selenium, pandas)  
**Output:** `data/brics_mpc_statements.csv`  
**Feeds into:** Project 5 — Monetary Policy Sentiment Analysis

---

### Motivation

The BRICS economies — Brazil, Russia, India, China, and South Africa — represent a significant and growing share of global trade and output. Yet their monetary policy frameworks, communication styles, and responses to shared shocks like currency fragmentation and trade disruptions diverge considerably. Tracking how BRICS central banks communicate policy over time provides a rich empirical lens on monetary divergence within this bloc.

This project constructs a structured, machine-readable dataset of monetary policy statements from BRICS central banks. The dataset is the foundation for Project 5, which applies sentiment analysis and topic modelling to quantify how hawkish or dovish communication has evolved across these institutions.

---

### Data Collection Strategy

The original goal was to collect MPC statements from all five BRICS central banks. In practice, data availability and accessibility constraints meant that four of the five could be covered:

| Country | Institution | Method | Coverage |
|---------|-------------|--------|----------|
| India | RBI | Direct scrape | 2016–2026 |
| Russia | CBR | Direct scrape | 2018–2026 |
| South Africa | SARB | Selenium scrape | 2006–2026 |
| China | PBOC | BIS CBSPEECHES | 1996–2025 |
| Brazil | BCB | — | Excluded |

> **Note on SARB:** The SARB website blocks standard HTTP requests at the network level. Selenium — a browser automation tool that drives a real Chrome instance — bypasses this restriction by rendering JavaScript before extraction.

> **Note on PBOC:** The PBOC does not publish clean post-meeting press releases in English. The BIS CBSPEECHES database, which aggregates central bank communications from over 100 institutions since 1996, is used instead — consistent with academic practice in cross-country central bank text analysis.

> **Note on BCB:** The Banco Central do Brasil (BCB) publishes COPOM statements through an open API, but only in Portuguese. The English website is JavaScript-rendered with no accessible endpoint returning English text. Machine translation was considered but rejected on grounds of introducing noise into the sentiment analysis in Project 5. BCB is therefore excluded, and the dataset covers the remaining four BRICS central banks.

---

### Final Output Schema

| Column | Description |
|--------|-------------|
| `country` | Country name |
| `central_bank` | Institution name |
| `date` | Publication date |
| `title` | Statement title |
| `text` | Full statement text |
| `url` | Source URL |

**Canonical corpus:** 302 communications in `data/brics_mpc_statements_v2.csv`. See README for exact counts, broader PBOC coverage, failed-document limitations and date precision. The earlier 305-row file is an intermediate with duplicate RBI records.


## Setup

In [ ]:
import sys
import pandas as pd

sys.path.append('.')
pd.set_option('display.max_colwidth', 100)

print("Ready.")

## 1. Reserve Bank of India (RBI)

The RBI publishes a **Governor's Statement** after each MPC meeting — the formal post-decision communication announcing the rate decision and policy stance. These are archived at `rbi.org.in` with a clean year-based URL structure.

India formally adopted inflation targeting in 2016 when the MPC was constituted, so coverage begins that year. The RBI holds 6 MPC meetings per year, the saved HTML extract contains 66 rows but only six unique documents; 57 historical PDFs supplement these in the final corpus.

The scraper (`scrapers/rbi_scraper.py`) loops through each year's archive page, identifies Governor's Statement links, fetches each page, and extracts the full text from the `text1` div.

In [ ]:
from scrapers.rbi_scraper import scrape_rbi

rbi_results = scrape_rbi(start_year=2016, end_year=2026)

In [ ]:
df_rbi = pd.DataFrame(rbi_results)

print(f"Statements scraped: {len(df_rbi)}")
print(f"Date range: 2016 → 2026 ({len(df_rbi)} statements)")
print(f"\nSample entry:\n")
df_rbi.head(3)

In [ ]:
df_rbi.to_csv("data/rbi_statements.csv", index=False)
print("Saved → data/rbi_statements.csv")

## 2. Bank of Russia (CBR)

The CBR publishes a **key rate press release** after each Board of Directors meeting — a concise statement announcing the rate decision and policy rationale. These are available in English at `cbr.ru/eng/press/pr/`.

The CBR adopted its current key rate framework in September 2013. Press releases from the CBR calendar page (`cbr.ru/eng/dkp/cal_mp/`) are available from 2018 onwards in a consistently structured format, giving us 51 statements through 2026.

> **Note on post-2022 data:** Bank of Russia statements from 2022 onwards are included but interpreted with awareness of the geopolitical context affecting Russia's external communications. This is noted transparently in line with standard academic practice.

The scraper (`scrapers/cbr_scraper.py`) fetches all press release links from the calendar page, then retrieves the full text from the `landing-text` div on each individual statement page.

In [ ]:
from scrapers.cbr_scraper import scrape_cbr

cbr_results = scrape_cbr()

In [ ]:
df_cbr = pd.DataFrame(cbr_results)

print(f"Statements scraped: {len(df_cbr)}")
print(f"Date range: {df_cbr['date'].iloc[-1]} → {df_cbr['date'].iloc[0]}")
print(f"\nSample entry:\n")
df_cbr.head(3)

In [ ]:
df_cbr.to_csv("data/cbr_statements.csv", index=False)
print("Saved → data/cbr_statements.csv")

## 3. South African Reserve Bank (SARB)

The SARB publishes **MPC statements** after each Monetary Policy Committee meeting — formal communications announcing the rate decision and economic outlook. Statements are available at `resbank.co.za` going back to 2006, giving us 124 statements.

The SARB website blocks standard HTTP requests at the network level. This scraper uses **Selenium**, a browser automation tool that drives a real Chrome instance in headless mode, bypassing the anti-bot protection. This is a standard technique for scraping JavaScript-rendered content.

The scraper (`scrapers/sarb_scraper.py`) navigates through 5 pages of the MPC statements index, collects all 124 URLs, then visits each page and extracts the full text by combining all `cmp-text` content blocks.

In [ ]:
from scrapers.sarb_scraper import scrape_sarb

sarb_results = scrape_sarb()

In [ ]:
df_sarb = pd.DataFrame(sarb_results)

print(f"Statements scraped: {len(df_sarb)}")
print(f"Date range: {df_sarb['date'].iloc[-1]} → {df_sarb['date'].iloc[0]}")
print(f"\nSample entry:\n")
df_sarb.head(3)

In [ ]:
df_sarb.to_csv("data/sarb_statements.csv", index=False)
print("Saved → data/sarb_statements.csv")

## 4. People's Bank of China (PBOC)

The PBOC does not publish clean post-meeting press releases in English comparable to the other central banks in this dataset. English-language communications are sparse and inconsistently structured on the PBOC website.

For this reason, PBOC data is sourced from the **BIS CBSPEECHES database** (`bis.org/cbspeeches/download.htm`), which aggregates central bank communications from over 100 central banks since 1996. The database contains 131 PBOC entries — speeches and statements by PBOC governors — covering 1996 to 2025.

This approach is consistent with academic practice. Several published papers using cross-country central bank text analysis use the BIS database as their primary source for PBOC communications.

> **Note on BCB:** The Banco Central do Brasil API provides COPOM statements only in Portuguese, and the English website is JavaScript-rendered with no accessible endpoint. BCB is therefore excluded from this dataset. The retained sources differ in document type and date coverage; direct comparisons require caution.

In [ ]:
from pathlib import Path
import zipfile
import os

# Load BIS speeches dataset
zip_path = Path("data/raw/speeches.zip")
extract_path = Path("data/raw/speeches")

# Unzip if not already done
if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)

df_bis = pd.read_csv(extract_path / "speeches.csv")
print(f"BIS dataset loaded: {df_bis.shape[0]} total speeches")

In [ ]:
# Filter for PBOC entries
pboc_mask = df_bis['description'].str.contains("People's Bank of China|PBC|PBOC",
                                                case=False, na=False)

df_pboc = df_bis[pboc_mask].copy()
df_pboc = df_pboc[['description', 'date', 'text', 'url']].copy()
df_pboc.rename(columns={"description": "title"}, inplace=True)
df_pboc.insert(0, 'country', 'China')
df_pboc.insert(1, 'central_bank', 'PBOC')
df_pboc['date'] = pd.to_datetime(df_pboc['date']).dt.strftime('%B %d, %Y')

print(f"PBOC statements: {len(df_pboc)}")
print(f"Date range: {df_pboc['date'].iloc[-1]} → {df_pboc['date'].iloc[0]}")
df_pboc.head(3)

In [ ]:
df_pboc.to_csv("data/pboc_statements.csv", index=False)
print("Saved → data/pboc_statements.csv")

## 5. Master Dataset

The following cells preserve the original intermediate assembly. Use build_corpus.py below to deduplicate RBI and add historical PDFs; Project 5 consumes the resulting v2 CSV.

| Central Bank | Country | Source | Statements |
|---|---|---|---|
| RBI | India | Direct scrape | 66 |
| CBR | Russia | Direct scrape | 51 |
| SARB | South Africa | Selenium scrape | 57 |
| PBOC | China | BIS CBSPEECHES | 131 |

In [ ]:
# Load all four individual CSVs
df_rbi = pd.read_csv("data/rbi_statements.csv")
df_cbr = pd.read_csv("data/cbr_statements.csv")
df_sarb = pd.read_csv("data/sarb_statements.csv")
df_pboc = pd.read_csv("data/pboc_statements.csv")

# Combine into master dataset
df_master = pd.concat([df_rbi, df_cbr, df_sarb, df_pboc], ignore_index=True)

print(f"Total statements: {len(df_master)}")
print(f"\nBy central bank:")
print(df_master['central_bank'].value_counts())
print(f"\nColumns: {list(df_master.columns)}")

In [ ]:
df_master.to_csv("data/brics_mpc_statements.csv", index=False)
print("Saved → data/brics_mpc_statements.csv")
print(f"\nProject 4 complete — {len(df_master)} statements across 4 central banks.")

In [ ]:
# Offline final assembly from saved extracts, including RBI historical PDFs.
from build_corpus import build_corpus
corpus = build_corpus()
corpus.to_csv("data/brics_mpc_statements_v2.csv", index=False)
print(corpus.groupby("central_bank").size())
